# Chatbot GoodWe — FIAP EV Challenge 2026
**RAG (Retrieval-Augmented Generation)** com LangChain + OpenAI

Fontes de conhecimento:
- PDFs do carregador GoodWe HCA G2 (manuais, datasheets, certificados)
- Vídeo de instalação (transcrição via Whisper — opcional)
- Site oficial GoodWe Brasil (todas as páginas)

> **Sem configuração necessária!** Os arquivos são baixados automaticamente
> de uma pasta pública do Google Drive. Qualquer pessoa pode rodar este notebook.

## 1. Instalação de Dependências

In [ ]:
# Instala e atualiza todas as bibliotecas necessárias para o projeto
%pip install --quiet --upgrade \
    gdown \
    langchain \
    langchain-core \
    langchain-community \
    langchain-openai \
    langchain-text-splitters \
    langgraph \
    pypdf \
    unstructured \
    openai \
    langsmith \
    python-dotenv \
    lxml \
    beautifulsoup4 \
    pandas



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 34.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.6/399.6 kB 16.3 MB/s eta 0:00:00
   ━

## 1.1. Importar pandas

In [ ]:
import pandas as pd
df = pd.read_json('conversas_chatbot.jsonl', lines=True)

## 2. Baixar Arquivos do Google Drive Público

A pasta do projeto é **pública** — qualquer pessoa pode rodar este notebook
sem precisar fazer login ou montar o Drive.

Os arquivos são baixados automaticamente para o ambiente do Colab.

| Pasta | Link público |
|---|---|
| PDFs | https://drive.google.com/drive/folders/1yp8TNGDGUQSx93c2G9BQ8OmwZkabONVv |
| Vídeos | https://drive.google.com/drive/folders/1pXtSf9HXu8aQA2f2-zgcyISWXUNO32sc |

In [ ]:
# Garante que o gdown está instalado antes de qualquer import
# Necessário caso esta célula seja rodada antes da Célula 1 de instalação
%pip install --quiet --upgrade gdown

import os     # módulo para criar pastas e trabalhar com caminhos de arquivos
import glob   # módulo para buscar arquivos usando padrões (ex: '*.pdf')
import gdown  # biblioteca para baixar arquivos e pastas do Google Drive por link público


# ---------------------------------------------------------------
# IDs das pastas públicas do Google Drive
# ---------------------------------------------------------------
# O ID é a parte do link após '/folders/' e antes de '?'
# Exemplo de link: https://drive.google.com/drive/folders/ID_AQUI?usp=sharing
#
# ⚠️  NÃO É NECESSÁRIO ALTERAR NADA ABAIXO — os IDs já estão configurados
#     Apenas execute a célula e aguarde o download

ID_PASTA_PDF     = '1yp8TNGDGUQSx93c2G9BQ8OmwZkabONVv'  # pasta com manuais e datasheets do HCA G2
ID_PASTA_VIDEO   = '1pXtSf9HXu8aQA2f2-zgcyISWXUNO32sc'  # pasta com vídeos de instalação
ID_PASTA_DESAFIO = '12hwiM0f3AX-mYhaSh-ppcShyxZGstQ4m'  # pasta com documentos do desafio FIAP/GoodWe


# ---------------------------------------------------------------
# Caminhos locais dentro do Colab onde os arquivos serão salvos
# ---------------------------------------------------------------
# /content/ é a pasta raiz do ambiente Colab (temporária — some ao fechar a sessão)
PASTA_PDF     = '/content/PDF'      # pasta local para os manuais e datasheets em PDF
PASTA_VIDEO   = '/content/Video'    # pasta local para os vídeos MP4
PASTA_DESAFIO = '/content/Desafio'  # pasta local para os documentos do desafio

# Cria as pastas locais caso ainda não existam
# exist_ok=True → não gera erro se a pasta já existir
os.makedirs(PASTA_PDF,     exist_ok=True)  # cria /content/PDF/
os.makedirs(PASTA_VIDEO,   exist_ok=True)  # cria /content/Video/
os.makedirs(PASTA_DESAFIO, exist_ok=True)  # cria /content/Desafio/


# ---------------------------------------------------------------
# Função auxiliar de download
# ---------------------------------------------------------------
# Criada para evitar repetição de código nas 3 chamadas de download abaixo
def baixar_pasta(id_drive: str, destino: str, nome: str):
    print(f'\nBaixando {nome}...')
    print(f'  Destino: {destino}')
    gdown.download_folder(
        id=id_drive,
        output=destino,
        quiet=False,
        use_cookies=False,
    )
    print(f'✓ {nome} baixado(s)!')


# ---------------------------------------------------------------
# Executa os downloads das 3 pastas
# ---------------------------------------------------------------
baixar_pasta(ID_PASTA_PDF,     PASTA_PDF,     'PDFs (manuais e datasheets)')
baixar_pasta(ID_PASTA_VIDEO,   PASTA_VIDEO,   'Vídeos')
baixar_pasta(ID_PASTA_DESAFIO, PASTA_DESAFIO, 'Documentos do Desafio FIAP/GoodWe')


# ---------------------------------------------------------------
# Verificação do que foi baixado
# ---------------------------------------------------------------
# glob com ** e recursive=True busca em subpastas também
# (o gdown às vezes cria uma subpasta com o nome da pasta do Drive)
arquivos_pdf     = sorted(glob.glob(os.path.join(PASTA_PDF,     '**', '*.pdf'),  recursive=True))
arquivos_video   = sorted(glob.glob(os.path.join(PASTA_VIDEO,   '**', '*.mp4'),  recursive=True))
arquivos_desafio_pdf  = sorted(glob.glob(os.path.join(PASTA_DESAFIO, '**', '*.pdf'),  recursive=True))
arquivos_desafio_docx = sorted(glob.glob(os.path.join(PASTA_DESAFIO, '**', '*.docx'), recursive=True))

# Exibe o relatório completo do que foi encontrado após os downloads
print('\n========== Arquivos baixados ==========')

print(f'\nPDFs — manuais/datasheets ({len(arquivos_pdf)} arquivo(s)):')
for f in arquivos_pdf:
    print(f'  • {os.path.basename(f)}')  # exibe só o nome, sem o caminho completo

print(f'\nVídeos ({len(arquivos_video)} arquivo(s)):')
for f in arquivos_video:
    print(f'  • {os.path.basename(f)}')

print(f'\nDesafio — PDFs ({len(arquivos_desafio_pdf)} arquivo(s)):')
for f in arquivos_desafio_pdf:
    print(f'  • {os.path.basename(f)}')

print(f'\nDesafio — DOCX ({len(arquivos_desafio_docx)} arquivo(s)):')
for f in arquivos_desafio_docx:
  print(f'  • {os.path.basename(f)}')

print('\n=======================================')


Baixando PDFs (manuais e datasheets)...
  Destino: /content/PDF


Retrieving folder contents


Processing file 1aBNEX3ZyofNv1KnbhKFF8k7vAcb6gKy7 GW_HCA G2_Certificate-EN1803.pdf
Processing file 1vs8uLeqJr3W_n2qH2VNznK0eyAV7B49g GW_HCA G2_Certificate-RETIE.pdf
Processing file 1KAg59CyIpngwVlETVny62BS1jdOrGyad GW_HCA G2_Datasheet-AU.pdf
Processing file 1Tp1Yywr4XJQz4ffKPkXk2Me5CsNYREZr GW_HCA G2_Datasheet-EN.pdf
Processing file 1NtE_6mabdKgHOnWlZuJrt7bkqNszHwgp GW_HCA G2_Datasheet-PT.pdf
Processing file 1cHGR5oXnZS_tZ6xIX8SXjGrgu4yiMqgs GW_HCA G2_Datasheet-TR.pdf
Processing file 1r8QUdFgZkh-80aJJQRHyyN1GpTPTnf27 GW_HCA G2_Datasheet_EMEA-EN.pdf
Processing file 1MO-9TnpKcbBnf1nsJlFmhv3m2tHrftQN GW_HCA G2_Datasheet_LATAM-ES.pdf
Processing file 1XpSUZhUKlaBJ5L8gvlmVx12jg8c2c429 GW_HCA G2_Ficha técnica_EMEA-ES.pdf
Processing file 163r_BlG-sJ-jCdAAPeNJOzzco9CRyOog GW_HCA G2_Manual de usuario-ES.pdf
Processing file 12NtTpT3xYlm0c3Q6PDWwVL9EXyYEeo7J GW_HCA G2_Quick Installation Guide-POLY.pdf
Processing file 1tKnv_op4w90U-Q1WNbHGdzaAHSRifKKF GW_HCA G2_User Manual-EN.pdf
Processing file 12

Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1aBNEX3ZyofNv1KnbhKFF8k7vAcb6gKy7
To: /content/PDF/GW_HCA G2_Certificate-EN1803.pdf
100%|██████████| 577k/577k [00:00<00:00, 35.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1vs8uLeqJr3W_n2qH2VNznK0eyAV7B49g
To: /content/PDF/GW_HCA G2_Certificate-RETIE.pdf
100%|██████████| 488k/488k [00:00<00:00, 10.0MB/s]
Downloading...
From: https://drive.google.com/uc?id=1KAg59CyIpngwVlETVny62BS1jdOrGyad
To: /content/PDF/GW_HCA G2_Datasheet-AU.pdf
100%|██████████| 414k/414k [00:00<00:00, 12.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Tp1Yywr4XJQz4ffKPkXk2Me5CsNYREZr
To: /content/PDF/GW_HCA G2_Datasheet-EN.pdf
100%|██████████| 415k/415k [00:00<00:00, 19.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1NtE_6mabdKgHOnWlZuJrt7bkqNszHwgp
To: /content/PDF/GW_HCA G2_Datasheet-PT.pdf
100%|██████████| 423k/423k [00

✓ PDFs (manuais e datasheets) baixado(s)!

Baixando Vídeos...
  Destino: /content/Video
Processing file 1c_50AI_U5bzPJxE4Ane0ndxx1HA_YmMa 15.GoodWeHCAG2SeriesUnboxingandInstallation_x264.mp4


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1c_50AI_U5bzPJxE4Ane0ndxx1HA_YmMa
To: /content/Video/15.GoodWeHCAG2SeriesUnboxingandInstallation_x264.mp4
100%|██████████| 43.4M/43.4M [00:00<00:00, 93.4MB/s]
Download completed
Retrieving folder contents


✓ Vídeos baixado(s)!

Baixando Documentos do Desafio FIAP/GoodWe...
  Destino: /content/Desafio
Processing file 1CJBJRl6qu0BhfTeUild1SlXQq2ifyTso 1CC - EV CHALLENGE GOODWE+FIAP 2026.pdf
Processing file 17YZPNJr3gmbZCBXlF2b8qVH9CEOe_55N Apresentação Challenge FIAP4+GoodWe 2026.pdf
Processing file 1GauSiF8LrCzXJxW8YyAXZRtEn-hOZFAU FIAP_EV Challenge_2026_Mentoria1.pdf
Processing file 1eR0ScF9ueJhycbw5-j7vNKmqgolcozo- Sprint 1 AI.docx


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1CJBJRl6qu0BhfTeUild1SlXQq2ifyTso
To: /content/Desafio/1CC - EV CHALLENGE GOODWE+FIAP 2026.pdf
100%|██████████| 1.64M/1.64M [00:00<00:00, 50.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=17YZPNJr3gmbZCBXlF2b8qVH9CEOe_55N
To: /content/Desafio/Apresentação Challenge FIAP4+GoodWe 2026.pdf
100%|██████████| 7.29M/7.29M [00:00<00:00, 65.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1GauSiF8LrCzXJxW8YyAXZRtEn-hOZFAU
To: /content/Desafio/FIAP_EV Challenge_2026_Mentoria1.pdf
100%|██████████| 569k/569k [00:00<00:00, 14.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1eR0ScF9ueJhycbw5-j7vNKmqgolcozo-
To: /content/Desafio/Sprint 1 AI.docx
100%|██████████| 15.8k/15.8k [00:00<00:00, 24.2MB/s]

✓ Documentos do Desafio FIAP/GoodWe baixado(s)!

========== Arquivos baixados ==========

PDFs — manuais/datasheets (17 arquivo(s)):
  • GW_HCA G2_Certificate-EN1803.pdf
  • GW_HCA G2_Certificate-RETIE.pdf
  • GW_HCA G2_Datasheet-AU.pdf
  • GW_HCA G2_Datasheet-EN.pdf
  • GW_HCA G2_Datasheet-PT.pdf
  • GW_HCA G2_Datasheet-TR.pdf
  • GW_HCA G2_Datasheet_EMEA-EN.pdf
  • GW_HCA G2_Datasheet_LATAM-ES.pdf
  • GW_HCA G2_Ficha técnica_EMEA-ES.pdf
  • GW_HCA G2_Manual de usuario-ES.pdf
  • GW_HCA G2_Quick Installation Guide-POLY.pdf
  • GW_HCA G2_User Manual-EN.pdf
  • GW_HCA G2_User Manual-PT.pdf
  • GW_HCA Series EV Charger_EU Declaration of Conformity_POLY.pdf
  • GW_HCA_HCA G2_Certificate-RED-AK.pdf
  • GW_HCA_HCA G2_Certificate-RED-COC.pdf
  • GW_HCA， HCA G2_Certificate-EN303645.pdf

Vídeos (1 arquivo(s)):
  • 15.GoodWeHCAG2SeriesUnboxingandInstallation_x264.mp4

Desafio — PDFs (3 arquivo(s)):
  • 1CC - EV CHALLENGE GOODWE+FIAP 2026.pdf
  • Apresentação Challenge FIAP4+GoodWe 2026.pdf
  • 


Download completed


## 3. Configurar Chaves de API

Vá em **Colab → ícone de chave 🔑 na barra lateral → Secrets** e adicione:
- `OPENAI_API_KEY` → sua chave da OpenAI (começa com `sk-...`)
- `LANGSMITH_API_KEY` → sua chave do LangSmith (em [smith.langchain.com](https://smith.langchain.com))

In [ ]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY']    = userdata.get('OPENAI_API_KEY')


os.environ['LANGSMITH_API_KEY'] = userdata.get('LANGSMITH_API_KEY')


os.environ['LANGCHAIN_TRACING_V2'] = 'true'

print('✓ Chaves de API configuradas com sucesso!')

✓ Chaves de API configuradas com sucesso!


## 4. Inicializar Modelo e Embeddings

In [ ]:
from langchain.chat_models import init_chat_model      # função que inicializa qualquer modelo de chat compatível com LangChain
from langchain_openai import OpenAIEmbeddings          # classe para gerar embeddings (vetores numéricos) via API da OpenAI
from langchain_core.vectorstores import InMemoryVectorStore  # banco de dados vetorial que fica na memória RAM (sem servidor externo)


llm = init_chat_model('gpt-4o-mini', model_provider='openai')


embeddings = OpenAIEmbeddings(model='text-embedding-3-small')


vector_store = InMemoryVectorStore(embeddings)

print('✓ Modelo e embeddings prontos!')

✓ Modelo e embeddings prontos!


## 5. Carregar Documentos

### 5a. PDFs — carregamento automático de todos os arquivos baixados

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

docs_pdf = []
if not arquivos_pdf:
    # Avisa o usuário que nenhum PDF foi baixado
    print('⚠️  Nenhum PDF encontrado. Verifique se o download da Célula 2 foi concluído.')
else:
    print(f'Carregando {len(arquivos_pdf)} PDF(s)...')

    # Percorre cada arquivo PDF encontrado após o download
    for caminho in arquivos_pdf:
        try:
            loader = PyPDFLoader(caminho)
            docs_pdf.extend(loader.load())
            print(f'  ✓ {os.path.basename(caminho)}')
        except Exception as e:
            print(f'  ✗ Erro em {os.path.basename(caminho)}: {e}')

print(f'\n✓ Total de páginas carregadas dos PDFs: {len(docs_pdf)}')

Carregando 17 PDF(s)...
  ✓ GW_HCA G2_Certificate-EN1803.pdf
  ✓ GW_HCA G2_Certificate-RETIE.pdf
  ✓ GW_HCA G2_Datasheet-AU.pdf
  ✓ GW_HCA G2_Datasheet-EN.pdf
  ✓ GW_HCA G2_Datasheet-PT.pdf
  ✓ GW_HCA G2_Datasheet-TR.pdf
  ✓ GW_HCA G2_Datasheet_EMEA-EN.pdf
  ✓ GW_HCA G2_Datasheet_LATAM-ES.pdf
  ✓ GW_HCA G2_Ficha técnica_EMEA-ES.pdf
  ✓ GW_HCA G2_Manual de usuario-ES.pdf
  ✓ GW_HCA G2_Quick Installation Guide-POLY.pdf
  ✓ GW_HCA G2_User Manual-EN.pdf
  ✓ GW_HCA G2_User Manual-PT.pdf
  ✓ GW_HCA Series EV Charger_EU Declaration of Conformity_POLY.pdf
  ✓ GW_HCA_HCA G2_Certificate-RED-AK.pdf
  ✓ GW_HCA_HCA G2_Certificate-RED-COC.pdf
  ✓ GW_HCA， HCA G2_Certificate-EN303645.pdf

✓ Total de páginas carregadas dos PDFs: 268


### 5b. Documentos do Desafio FIAP/GoodWe (PDFs + DOCX)

Carrega os arquivos de briefing, apresentações e especificações do desafio para que o chatbot entenda o contexto do projeto.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader    # loader para arquivos PDF
from langchain_community.document_loaders import Docx2txtLoader  # loader para arquivos Word (.docx)

docs_desafio = []  # lista vazia que vai acumular todos os documentos do desafio (PDFs + DOCX)


# ---------------------------------------------------------------
# Carrega os PDFs do desafio (apresentações, briefings, etc.)
# ---------------------------------------------------------------
if not arquivos_desafio_pdf:
    print('⚠️  Nenhum PDF do desafio encontrado.')  # avisa se a pasta estiver vazia
else:
    print(f'Carregando {len(arquivos_desafio_pdf)} PDF(s) do desafio...')

    for caminho in arquivos_desafio_pdf:         # percorre cada PDF encontrado na pasta Desafio
        try:
            loader = PyPDFLoader(caminho)         # cria um loader para o PDF atual
            docs_desafio.extend(loader.load())   # carrega todas as páginas e adiciona à lista
            print(f'  ✓ {os.path.basename(caminho)}')  # confirma que o arquivo foi carregado
        except Exception as e:
            print(f'  ✗ Erro em {os.path.basename(caminho)}: {e}')  # exibe erro e continua


# ---------------------------------------------------------------
# Carrega os arquivos Word (.docx) do desafio
# ---------------------------------------------------------------
if not arquivos_desafio_docx:
    print('⚠️  Nenhum DOCX do desafio encontrado.')  # avisa se nenhum .docx foi baixado
else:
    print(f'\nCarregando {len(arquivos_desafio_docx)} DOCX(s) do desafio...')

    for caminho in arquivos_desafio_docx:
        try:
            loader = Docx2txtLoader(caminho)

            docs_desafio.extend(loader.load())
            print(f'  ✓ {os.path.basename(caminho)}')
        except Exception as e:
            print(f'  ✗ Erro em {os.path.basename(caminho)}: {e}')


# Exibe o total de documentos carregados da pasta Desafio
print(f'\n✓ Total de documentos do desafio carregados: {len(docs_desafio)}')

Carregando 3 PDF(s) do desafio...
  ✓ 1CC - EV CHALLENGE GOODWE+FIAP 2026.pdf
  ✓ Apresentação Challenge FIAP4+GoodWe 2026.pdf
  ✓ FIAP_EV Challenge_2026_Mentoria1.pdf

Carregando 1 DOCX(s) do desafio...
  ✗ Erro em Sprint 1 AI.docx: No module named 'docx2txt'

✓ Total de documentos do desafio carregados: 48


### 5c. Site da GoodWe Brasil — Rastreamento Completo de Todas as Páginas

Usamos o `RecursiveUrlLoader` que começa pela página inicial e segue automaticamente
**todos os links internos** do site, carregando cada página encontrada.

**Como funciona a profundidade (`max_depth`):**
```
Profundidade 0 → br.goodwe.com/                      (página inicial)
Profundidade 1 → br.goodwe.com/produtos/             (links da página inicial)
Profundidade 2 → br.goodwe.com/produtos/ev-charger/  (links das páginas de produtos)
Profundidade 3 → br.goodwe.com/produtos/ev-charger/hca-g2/  (sub-páginas)
```

In [ ]:
import re                  # módulo de expressões regulares — usado para limpar espaços extras do texto extraído
import time                # módulo de tempo — usado para criar pausas entre as requisições ao site
from bs4 import BeautifulSoup                                        # biblioteca para analisar HTML e extrair apenas o texto útil
from langchain_community.document_loaders import RecursiveUrlLoader  # loader que rastreia todas as páginas do site seguindo os links


# ---------------------------------------------------------------
# Função extratora de texto — compartilhada entre todos os sites
# ---------------------------------------------------------------
def extrair_texto(html: str) -> str:
    soup = BeautifulSoup(html, 'lxml')

    for tag in soup.find_all(['script', 'style']):
        tag.decompose()
    for tag in soup.find_all(['nav', 'footer', 'header']):
        tag.decompose()

    texto = soup.get_text(separator='\n')
    texto = re.sub(r'\n{3,}', '\n\n', texto)
    return texto.strip()


docs_web = []  # lista unificada — acumula páginas de todos os sites


# ═══════════════════════════════════════════════════════════════════════════════
# FONTE 1 — Site GoodWe Brasil (rastreamento recursivo completo)
# ═══════════════════════════════════════════════════════════════════════════════
URL_GOODWE = 'https://br.goodwe.com/'

loader_goodwe = RecursiveUrlLoader(
    url=URL_GOODWE,

    max_depth=3,               # profundidade máxima de links a seguir a partir da URL base

    extractor=extrair_texto,   # limpa o HTML de cada página antes de salvar

    prevent_outside=True,      # nunca sai do domínio br.goodwe.com

    timeout=15,                # ignora páginas que demoram mais de 15 s

    check_response_status=True,  # ignora páginas com erro HTTP (404, 500, etc.)

    continue_on_failure=True,  # continua mesmo se uma página falhar

    exclude_dirs=(
        'https://br.goodwe.com/wp-content/',  # arquivos internos do WordPress (imagens, uploads)
        'https://br.goodwe.com/wp-admin/',    # painel de administração — área restrita
        'https://br.goodwe.com/cdn-cgi/',     # infraestrutura Cloudflare — sem conteúdo útil
        'https://br.goodwe.com/feed/',        # feed RSS — formato XML, não legível pelo chatbot
    ),
)

print(f'[1/3] Rastreando GoodWe Brasil: {URL_GOODWE}')
print(f'      Profundidade máxima: 3 níveis | Isso pode levar alguns minutos...')
print('-' * 60)

n_goodwe = 0
for i, doc in enumerate(loader_goodwe.lazy_load()):
    docs_web.append(doc)
    n_goodwe += 1
    print(f'  [{i + 1:>3}] ✓ {doc.metadata.get("source", "URL desconhecida")}')
    time.sleep(0.3)

print(f'✓ GoodWe Brasil: {n_goodwe} página(s) carregada(s)\n')


# ═══════════════════════════════════════════════════════════════════════════════
# FONTE 2 — Páginas diretas do produto HCA G2 (certificações e especificações)
# ═══════════════════════════════════════════════════════════════════════════════


URLS_HCA_G2 = [
    'https://br.goodwe.com/ev-charger/',                    # página geral de EV Charger no site BR
    'https://www.goodwe.com/ev-charger/hca-g2',             # página do produto no site global (EN)
    'https://en.goodwe.com/ev-charger/hca-g2',              # subdomínio em inglês — pode ter specs
    'https://www.goodwe.com/products/ev-charger',           # listagem de produtos EV no site global
]

print('[2/3] Carregando páginas diretas do HCA G2 (certificações e specs)...')
print('-' * 60)

n_hca_g2 = 0
for url in URLS_HCA_G2:
    try:
        loader_direto = RecursiveUrlLoader(
            url=url,
            max_depth=1,           # carrega SÓ esta página — sem seguir links
            extractor=extrair_texto,
            timeout=15,
            check_response_status=True,
            continue_on_failure=True,
        )
        for doc in loader_direto.lazy_load():
            docs_web.append(doc)
            n_hca_g2 += 1
            print(f'  ✓ {doc.metadata.get("source", url)}')
        time.sleep(0.5)
    except Exception as e:
        print(f'  ✗ {url}: {e}')  # ignora URLs que não existem ou retornam erro

print(f'✓ HCA G2 direto: {n_hca_g2} página(s) carregada(s)\n')


# ═══════════════════════════════════════════════════════════════════════════════
# FONTE 3 — Open Charge Alliance — Protocolo OCPP
# ═══════════════════════════════════════════════════════════════════════════════


URL_OCPP = 'https://openchargealliance.org/protocols/open-charge-point-protocol/'

loader_ocpp = RecursiveUrlLoader(
    url=URL_OCPP,

    max_depth=2,               # página do protocolo + subpáginas de versão
                               # evita rastrear o site inteiro da organização

    extractor=extrair_texto,   # mesma função de limpeza usada nos outros loaders

    prevent_outside=True,      # permanece dentro do domínio openchargealliance.org

    timeout=20,                # site internacional — pode ser mais lento

    check_response_status=True,
    continue_on_failure=True,
)

print(f'[3/3] Rastreando Open Charge Alliance (OCPP): {URL_OCPP}')
print(f'      Profundidade: 2 níveis (protocolo + versões OCPP 1.6 / 2.0.1 / ...)')
print('-' * 60)

n_ocpp = 0
for i, doc in enumerate(loader_ocpp.lazy_load()):
    docs_web.append(doc)
    n_ocpp += 1
    print(f'  [{i + 1:>3}] ✓ {doc.metadata.get("source", "URL desconhecida")}')
    time.sleep(0.3)


# ---------------------------------------------------------------
# Relatório final consolidado
# ---------------------------------------------------------------
print('-' * 60)
print('✓ Rastreamento web concluído!')
print(f'  ├─ [1] GoodWe Brasil (recursivo):  {n_goodwe:>3} página(s)')
print(f'  ├─ [2] HCA G2 direto (specs):      {n_hca_g2:>3} página(s)')
print(f'  └─ [3] Open Charge Alliance (OCPP):{n_ocpp:>3} página(s)')
print(f'  {"─" * 38}')
print(f'     TOTAL páginas web:              {len(docs_web):>3} página(s)')


[1/3] Rastreando GoodWe Brasil: https://br.goodwe.com/
      Profundidade máxima: 3 níveis | Isso pode levar alguns minutos...
------------------------------------------------------------
  [  1] ✓ https://br.goodwe.com/
  [  2] ✓ https://br.goodwe.com/commercial-and-industrial-energy-storage-solution
  [  3] ✓ https://br.goodwe.com/faqs
  [  4] ✓ https://br.goodwe.com/imprint


  [  5] ✓ https://br.goodwe.com/case-study
  [  6] ✓ https://br.goodwe.com/news
  [  7] ✓ https://br.goodwe.com/products
  [  8] ✓ https://br.goodwe.com/goodwe-plus-customer-program
  [  9] ✓ https://br.goodwe.com/data-protection
  [ 10] ✓ https://br.goodwe.com/usinas-solares-linha-ht-225-250kw-trifasico-goodwe
  [ 11] ✓ https://br.goodwe.com/solutions
  [ 12] ✓ https://br.goodwe.com/join-us
  [ 13] ✓ https://br.goodwe.com/products?tp=4
  [ 14] ✓ https://br.goodwe.com/cases
  [ 15] ✓ https://br.goodwe.com/career
  [ 16] ✓ https://br.goodwe.com/company-profile


  [ 17] ✓ https://br.goodwe.com/utility-scale-solar-pv-solution
  [ 18] ✓ https://br.goodwe.com/products?tp=1
  [ 19] ✓ https://br.goodwe.com/company-history
  [ 20] ✓ https://br.goodwe.com/residential-rooftop
  [ 21] ✓ https://br.goodwe.com/distribuidores
  [ 22] ✓ https://br.goodwe.com/solar-system-quality-control
  [ 23] ✓ https://br.goodwe.com/ebooks
  [ 24] ✓ https://br.goodwe.com/amer
  [ 25] ✓ https://br.goodwe.com/products?tp=2
  [ 26] ✓ https://br.goodwe.com/Skippower/downloadFileF?id=3&mid=68
  [ 27] ✓ https://br.goodwe.com/products?tp=3
  [ 28] ✓ https://br.goodwe.com/download
  [ 29] ✓ https://br.goodwe.com/products?tp=7
  [ 30] ✓ https://br.goodwe.com/products?tp=6
  [ 31] ✓ https://br.goodwe.com/residential-solutions
  [ 32] ✓ https://br.goodwe.com/warranty
  [ 33] ✓ https://br.goodwe.com/energy-storage-solutions
  [ 34] ✓ https://br.goodwe.com/inversores-comercial-linha-lvmt-goodwe
  [ 35] ✓ https://br.goodwe.com/products?tp=5
  [ 36] ✓ https://br.goodwe.com/inversores-c

  ✓ https://en.goodwe.com/ev-charger/hca-g2


✓ HCA G2 direto: 1 página(s) carregada(s)

[3/3] Rastreando Open Charge Alliance (OCPP): https://openchargealliance.org/protocols/open-charge-point-protocol/
      Profundidade: 2 níveis (protocolo + versões OCPP 1.6 / 2.0.1 / ...)
------------------------------------------------------------
  [  1] ✓ https://openchargealliance.org/protocols/open-charge-point-protocol/
  [  2] ✓ https://openchargealliance.org/about-us/
  [  3] ✓ https://openchargealliance.org/whitepapers/
  [  4] ✓ https://openchargealliance.org/protocols/
  [  5] ✓ https://openchargealliance.org/ocpp-plugfest/
  [  6] ✓ https://openchargealliance.org/download-ocpp/
  [  7] ✓ https://openchargealliance.org/wp-content/themes/smeders/css/main.min.css?ver=1.4
  [  8] ✓ https://openchargealliance.org/faq/
  [  9] ✓ https://openchargealliance.org/wp-json/
  [ 10] ✓ https://openchargealliance.org/terms-of-service/
  [ 11] ✓ https://openchargealliance.org/oca-i-chatbot/
  [ 12] ✓ https://openchargealliance.org/wp-json/oembed/

  [ 22] ✓ https://openchargealliance.org/?p=16
  [ 23] ✓ https://openchargealliance.org/protocols/archive/
  [ 24] ✓ https://openchargealliance.org/join-us/
  [ 25] ✓ https://openchargealliance.org/events/
  [ 26] ✓ https://openchargealliance.org/certification/certification-ocpp-2-0-1/
  [ 27] ✓ https://openchargealliance.org/community/
  [ 28] ✓ https://openchargealliance.org/ocpp-info-whitepapers/
  [ 29] ✓ https://openchargealliance.org/careers/
  [ 30] ✓ https://openchargealliance.org/privacy-policy/
  [ 31] ✓ https://openchargealliance.org/sign-in
  [ 32] ✓ https://openchargealliance.org/wp-content/plugins/complianz-gdpr/assets/css/cookieblocker.min.css?ver=1777534525
  [ 33] ✓ https://openchargealliance.org/wp-includes/css/dist/block-library/style.min.css?ver=6.9.4
  [ 34] ✓ https://openchargealliance.org/news/
  [ 35] ✓ https://openchargealliance.org/test-tool/
------------------------------------------------------------
✓ Rastreamento web concluído!
  ├─ [1] GoodWe Brasil (recu

### 5d. Base de Conhecimento FAQ — Complementar (respostas garantidas)

Documentos criados diretamente no código com informações **críticas** do HCA G2 que
o chatbot precisa responder corretamente, independentemente do que foi capturado
no web scraping ou nos PDFs.

**Para essa parte o ideal é criar um documento no Word e uma nova pasta ou criar um pdf e adicionar na pasta PDF, assim qualquer pessoa pode adicionar perguntas de FAQ facilmente na IA**

> **Por que isso é necessário?**  
> Sites podem bloquear scraping, conteúdo importante pode estar em JavaScript dinâmico
> (não visível ao BeautifulSoup), ou a informação pode estar em uma profundidade
> além do `max_depth` configurado. O FAQ garante cobertura mínima para perguntas-chave.

In [ ]:
from langchain_core.documents import Document  # importado aqui também pois esta célula pode rodar antes da Seção 8



CONHECIMENTO_COMPLEMENTAR = [

    # ── LED e notificação de fim de carregamento ─────────────────────────────
    {
        "titulo": "LED e notificação de fim de carregamento — HCA G2",
        "conteudo": """
GoodWe HCA G2 — Como saber quando o carro terminou de carregar

Indicações do LED durante a sessão:
- LED branco pulsante: aguardando conexão do veículo (standby)
- LED azul (fixo ou pulsante): carregamento em andamento
- LED verde fixo: carregamento CONCLUÍDO — bateria do veículo totalmente carregada
- LED vermelho: falha ou erro — consulte o manual para diagnóstico

Formas de identificar o fim da sessão de carregamento:
1. Indicador LED: quando a carga é concluída, o LED do HCA G2 muda para VERDE FIXO.
2. Notificação digital via OCPP: se o carregador estiver integrado a um aplicativo
   ou portal de gestão (CSMS — Charge Station Management System) via protocolo OCPP,
   o usuário recebe uma notificação digital com o resumo completo da sessão:
   - Energia total fornecida (kWh)
   - Duração total da sessão de carregamento
   - Custo da sessão (se o sistema de billing estiver configurado)
3. O próprio veículo: a maioria dos veículos elétricos para de aceitar carga
   automaticamente quando a bateria atinge 100%, o que coincide com a indicação
   de verde fixo no carregador.

O protocolo OCPP (Open Charge Point Protocol) permite que o HCA G2 envie
mensagens StatusNotification e StopTransaction para o sistema de gestão,
que pode acionar notificações push no smartphone do operador ou usuário.
        """.strip(),
    },

    # ── Certificações internacionais ─────────────────────────────────────────
    {
        "titulo": "Certificações internacionais do GoodWe HCA G2",
        "conteudo": """
GoodWe HCA G2 — Certificações Internacionais

O GoodWe HCA G2 possui as seguintes certificações:

- CE (Conformité Européenne): certifica conformidade com as diretivas de segurança
  elétrica e compatibilidade eletromagnética (EMC) da União Europeia. Obrigatória
  para comercialização em todos os países membros da UE.

- CB (IEC System for Mutual Recognition of Testing and Certification): certificação
  do sistema internacional IEC, reconhecida em mais de 50 países. Atesta conformidade
  com normas de segurança elétrica internacionais baseadas em padrões IEC.

- RCM (Regulatory Compliance Mark): certificação para Austrália e Nova Zelândia.
  Abrange segurança elétrica e compatibilidade eletromagnética conforme as normas
  australianas e neozelandesas.

- RETIE (Reglamento Técnico de Instalaciones Eléctricas): certificação colombiana
  exigida para instalações elétricas no mercado da Colômbia. Garante conformidade
  com as normas técnicas elétricas colombianas.

- EN 303 645: norma europeia de cibersegurança para dispositivos de consumo
  conectados à internet (IoT). Garante que o HCA G2 segue boas práticas de
  segurança cibernética, como proteção de credenciais e atualizações seguras.

Essas certificações atestam conformidade com normas de segurança elétrica,
compatibilidade eletromagnética e cibersegurança em múltiplos mercados globais.
        """.strip(),
    },

    # ── Protocolo OCPP e integração ──────────────────────────────────────────
    {
        "titulo": "Protocolo OCPP e integração do HCA G2 com sistemas de gestão",
        "conteudo": """
GoodWe HCA G2 — Protocolo OCPP e Integração com Sistemas de Gestão

O GoodWe HCA G2 suporta o protocolo OCPP (Open Charge Point Protocol),
desenvolvido pela Open Charge Alliance (OCA).

Versões suportadas: OCPP 1.6 e OCPP 2.0.1

O que é OCPP:
- Protocolo aberto de comunicação entre carregadores (EVSE) e sistemas de gestão (CSMS)
- Permite controle remoto, monitoramento em tempo real e billing das sessões
- Garante interoperabilidade entre carregadores de diferentes fabricantes

Funcionalidades habilitadas pelo OCPP no HCA G2:
- Início e encerramento remoto de sessões de carregamento
- Monitoramento em tempo real do status do carregador (Available, Charging, Faulted)
- Notificações de eventos: início, fim e falhas de sessão
- Autenticação de usuários via RFID ou token digital
- Gestão de energia e limite de potência remotamente
- Diagnóstico remoto e atualização de firmware (OTA)
- Relatórios de consumo energético por sessão para billing

Integração com aplicativos:
Quando o HCA G2 está conectado a um CSMS via OCPP, o operador pode receber
notificações no smartphone com resumo de cada sessão: energia fornecida (kWh),
duração, horário de início/fim e custo (se o billing estiver configurado).
        """.strip(),
    },
]

# ---------------------------------------------------------------
# Cria os objetos Document para indexação no banco vetorial
# ---------------------------------------------------------------
docs_faq = [
    Document(
        page_content=item["conteudo"],
        metadata={
            "source": f"FAQ — {item['titulo']}",  # identificador da fonte nos metadados
            "type":   "faq",                       # tipo personalizado para rastreamento
        }
    )
    for item in CONHECIMENTO_COMPLEMENTAR
]

print(f'✓ Base FAQ complementar: {len(docs_faq)} documento(s) criado(s)')
for doc in docs_faq:
    print(f'  • {doc.metadata["source"]}')

✓ Base FAQ complementar: 3 documento(s) criado(s)
  • FAQ — LED e notificação de fim de carregamento — HCA G2
  • FAQ — Certificações internacionais do GoodWe HCA G2
  • FAQ — Protocolo OCPP e integração do HCA G2 com sistemas de gestão


### 5e. Vídeo — Transcrição com Whisper (OPCIONAL)

> **Pule esta célula** se não quiser usar o conteúdo do vídeo.
> Para ativar: instale as dependências da Célula 1 (seção vídeo) e descomente o código abaixo.
>
> Os vídeos já foram baixados automaticamente na Célula 2.
> Se novos vídeos forem adicionados à pasta do Drive, basta rodar a Célula 2 novamente.

In [ ]:
# ============================================================
# ⬇️  OPCIONAL: transcrição dos vídeos com Whisper
#    Descomente e execute apenas se instalou os pacotes de vídeo
# ============================================================

docs_video = []  # lista vazia — será preenchida só se a transcrição for ativada abaixo

# # 'arquivos_video' já foi definido na Célula 2 com todos os MP4 baixados do Drive
# # Se forem adicionados novos vídeos no Drive, rode a Célula 2 novamente antes desta
#
# if not arquivos_video:
#     print('Nenhum vídeo MP4 encontrado.')  # avisa se nenhum vídeo foi baixado
# else:
#     import whisper                               # importa o Whisper da OpenAI para transcrição de áudio
#     from langchain_core.documents import Document  # classe para padronizar o texto transcrito como documento LangChain
#
#     # Carrega o modelo Whisper — escolha a variante pelo equilíbrio velocidade × precisão:
#     # 'tiny' (mais rápido, menos preciso) | 'base' | 'small' | 'medium' | 'large' (mais lento, mais preciso)
#     model_whisper = whisper.load_model('base')
#
#     # Percorre todos os vídeos baixados automaticamente na Célula 2
#     # Novos vídeos adicionados ao Drive serão incluídos automaticamente
#     for caminho_video in arquivos_video:
#         print(f'Transcrevendo: {os.path.basename(caminho_video)}')  # exibe o nome do vídeo sendo transcrito
#
#         # Transcreve o áudio do vídeo e força o idioma português
#         resultado = model_whisper.transcribe(caminho_video, language='pt')
#         transcricao = resultado['text']  # extrai só o texto do resultado da transcrição
#
#         # Cria um Document LangChain com o texto transcrito
#         docs_video.append(Document(
#             page_content=transcricao,  # o conteúdo do documento é o texto transcrito do vídeo
#             metadata={                 # metadados que identificam a origem do documento
#                 'source': os.path.basename(caminho_video),  # nome do arquivo de vídeo
#                 'tipo': 'video',       # marcador para identificar que veio de um vídeo
#             }
#         ))
#         print(f'  ✓ Concluído ({len(transcricao)} caracteres)')  # confirma e exibe o tamanho da transcrição
#
#     print(f'\nTotal de vídeos transcritos: {len(docs_video)}')

print(f'Célula de vídeo pulada (comentada). Documentos de vídeo: {len(docs_video)}')

Célula de vídeo pulada (comentada). Documentos de vídeo: 0


### 5final. Combinar todos os documentos
#### Junta em uma única lista todos os documentos carregados de todas as fontes
#### O operador + concatena listas em Python
docs = docs_pdf + docs_desafio + docs_web + docs_faq + docs_video

In [ ]:
docs = docs_pdf + docs_desafio + docs_web + docs_faq + docs_video

print('Resumo das fontes carregadas:')
print(f'  PDFs (manuais/datasheets): {len(docs_pdf)} páginas')
print(f'  Desafio FIAP/GoodWe:       {len(docs_desafio):>4} documento(s)')
print(f'  Sites web (GoodWe + OCPP): {len(docs_web):>4} página(s)')
print(f'  FAQ complementar:          {len(docs_faq):>4} documento(s)')
print(f'  Vídeo (transcrição):       {len(docs_video):>4} documento(s)')
print(f'  {"─" * 37}')
print(f'  TOTAL:                     {len(docs):>4} documentos')

Resumo das fontes carregadas:
  PDFs (manuais/datasheets): 268 páginas
  Desafio FIAP/GoodWe:         48 documento(s)
  Sites web (GoodWe + OCPP):   78 página(s)
  FAQ complementar:             3 documento(s)
  Vídeo (transcrição):          0 documento(s)
  ─────────────────────────────────────
  TOTAL:                      397 documentos


## 6. Processar e Indexar Documentos

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter  # divide textos longos em trechos menores

# Verifica se há documentos carregados antes de continuar
# Sem documentos, as próximas etapas não têm o que processar
if not docs:
    raise RuntimeError('Nenhum documento carregado! Verifique as células anteriores.')

# Configura o divisor de texto
# Os modelos de embedding têm limite de caracteres por vez, por isso dividimos em trechos
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # tamanho máximo de cada trecho em caracteres (≈ 200 palavras)
    chunk_overlap=200,    # os últimos 200 caracteres de um trecho são repetidos no início do próximo
                          # evita perda de contexto quando uma informação fica no limite entre trechos
    add_start_index=True, # salva a posição de início de cada trecho no documento original
)

# Divide todos os documentos em trechos menores
# Retorna uma nova lista onde cada elemento é um trecho de um documento original
all_splits = text_splitter.split_documents(docs)

print(f'✓ Documentos divididos em {len(all_splits)} trechos menores')  # exibe o total de trechos gerados

✓ Documentos divididos em 9296 trechos menores


In [ ]:
print('Indexando trechos no banco vetorial...')  # avisa que o processo começou
                                                 # esta etapa converte cada trecho em vetor via API da OpenAI
                                                 # e salva no banco vetorial — pode demorar dependendo da quantidade

# Converte cada trecho em vetor numérico (embedding) e armazena no banco vetorial
# Retorna a lista de IDs únicos atribuídos a cada trecho no banco
document_ids = vector_store.add_documents(documents=all_splits)

print(f'✓ {len(document_ids)} trechos indexados com sucesso!')  # confirma que todos foram armazenados

Indexando trechos no banco vetorial...
✓ 9296 trechos indexados com sucesso!


## 7. Configurar o Prompt RAG

In [ ]:
from langsmith import Client          # cliente da API do LangSmith para acessar o Hub de prompts
from google.colab import userdata     # módulo para ler os Secrets salvos no Colab

# Cria o cliente do LangSmith usando a chave de API salva nos Secrets
client_ls = Client(api_key=userdata.get('LANGSMITH_API_KEY'))

# Baixa o template de prompt RAG padrão do LangSmith Hub
# 'rlm/rag-prompt' é um prompt público e testado para tarefas de RAG
# include_model=True → traz também a configuração do modelo junto com o prompt
# dangerously_pull_public_prompt=True → confirma que confiamos neste prompt público
#   (necessário nas versões recentes do LangSmith por segurança — o Hub pode conter
#    objetos LangChain serializados de terceiros; como é um prompt oficial e conhecido,
#    é seguro habilitar esta flag)
prompt = client_ls.pull_prompt(
    'rlm/rag-prompt',
    include_model=True,
    dangerously_pull_public_prompt=True,  # ← flag obrigatória nas versões recentes do LangSmith
)

# Substitui o template padrão por um personalizado para o contexto GoodWe/FIAP em português
# messages[0] = primeira mensagem do prompt (system message)
# .prompt.template = o texto do template preenchido com {question} e {context} em tempo de execução
prompt.messages[0].prompt.template = (
    'Você é um assistente especializado no carregador veicular GoodWe HCA G2 '
    'e nos produtos GoodWe em geral. '
    'Você atende cinco perfis de usuário:'
    'Operadores comerciais de eletropostos (ChargeGrid Intelligence)'
    'Síndicos e administradores de condomínios (EV ChargeOps)'
    'Moradores e usuários finais do carregador'
    'Técnicos de instalação e manutenção'
    'GoodWe (monitoramento centralizado e gestão da frota de dispositivos alugados)'

    'Responda usando APENAS as informações do contexto fornecido. '
    'Não invente dados técnicos.'
    'Se não souber a resposta com base no contexto, diga '
    '"Não tenho essa informação nos documentos disponíveis". '
    'Responda sempre em português brasileiro de forma clara e objetiva.\n\n'
    'Para perguntas técnicas de instalação, diga'
    '"Consulte um eletricista habilitado e seguir as normas locais."'
    'Mantenha o contexto da conversa — '
    'use o histórico de mensagens para responder perguntas de acompanhamento.'
    'Pergunta: {question}\n'   # {question} será substituído pela pergunta real do usuário
    'Contexto: {context}\n'    # {context} será substituído pelos trechos recuperados do banco vetorial
    'Resposta:'
)

print('✓ Prompt RAG configurado!')

✓ Prompt RAG configurado!


## 8. Construir o Pipeline RAG (LangGraph)

In [ ]:
from langchain_core.documents import Document  # classe que representa um documento com conteúdo e metadados
from typing import List                        # tipo para indicar que uma variável é uma lista
from typing_extensions import TypedDict        # permite criar dicionários com tipos definidos para cada chave
from langgraph.graph import START, StateGraph  # ferramentas para construir o grafo de execução do pipeline


# Define a estrutura do estado que circula entre as etapas do pipeline
# Cada etapa lê campos deste dicionário e escreve novos campos nele
class State(TypedDict):
    question: str            # pergunta digitada pelo usuário
    context: List[Document]  # lista de trechos relevantes recuperados do banco vetorial
    answer: str              # resposta gerada pelo modelo de linguagem


def retrieve(state: State):
    # Etapa 1 do pipeline: busca semântica no banco vetorial
    # Converte a pergunta em vetor e encontra os trechos mais próximos semanticamente
    retrieved_docs = vector_store.similarity_search(
        state['question'],  # pergunta do usuário — será convertida em vetor para a busca
        k=4,                # retorna os 4 trechos mais relevantes encontrados no banco
    )
    return {'context': retrieved_docs}  # salva os trechos no estado para a próxima etapa


def generate(state: State):
    # Etapa 2 do pipeline: geração da resposta com base no contexto recuperado
    # Junta o conteúdo de todos os trechos em um único bloco de texto
    docs_content = '\n\n'.join(doc.page_content for doc in state['context'])

    # Preenche o template do prompt substituindo {question} e {context} pelos valores reais
    messages = prompt.invoke({'question': state['question'], 'context': docs_content})

    # Envia as mensagens ao modelo de linguagem e obtém a resposta gerada
    response = llm.invoke(messages)
    return {'answer': response.content}  # salva a resposta gerada no estado


# Monta o grafo de execução definindo a sequência: retrieve → generate
graph_builder = StateGraph(State).add_sequence([retrieve, generate])

# Define que o grafo sempre começa pela etapa 'retrieve' (busca)
graph_builder.add_edge(START, 'retrieve')

# Compila o grafo em um objeto executável
graph = graph_builder.compile()

print('✓ Pipeline RAG pronto!')

✓ Pipeline RAG pronto!


## 9. Testar uma Pergunta

###**P1**:  Qual protocolo o GoodWe HCA G2 suporta para integração com sistemas de gestão de eletropostos?

In [ ]:
# Mude a pergunta abaixo para testar diferentes assuntos sobre a GoodWe
pergunta_teste = 'Qual protocolo o GoodWe HCA G2 suporta para integração com sistemas de gestão de eletropostos?'

# Executa o pipeline completo: busca no banco vetorial → geração da resposta
resultado = graph.invoke({'question': pergunta_teste})

# Exibe a pergunta e a resposta gerada
print(f'Pergunta: {pergunta_teste}')
print(f'\nResposta: {resultado["answer"]}')

# Exibe de quais documentos vieram os trechos usados para gerar a resposta
print('\nFontes usadas para responder:')
for doc in resultado['context']:
    fonte  = doc.metadata.get('source', 'desconhecida')  # URL ou caminho do arquivo de origem
    pagina = doc.metadata.get('page', '')                # número da página (só para PDFs)
    print(f'  • {os.path.basename(str(fonte))} {"p." + str(pagina) if pagina != "" else ""}')

Pergunta: Qual protocolo o GoodWe HCA G2 suporta para integração com sistemas de gestão de eletropostos?

Resposta: O GoodWe HCA G2 suporta o protocolo OCPP (Open Charge Point Protocol), com versões OCPP 1.6 e OCPP 2.0.1. Esse protocolo permite controle remoto, monitoramento em tempo real e faturamento das sessões de carregamento, garantindo a interoperabilidade entre carregadores de diferentes fabricantes.

Fontes usadas para responder:
  • FAQ — Protocolo OCPP e integração do HCA G2 com sistemas de gestão 
  • FAQ — Protocolo OCPP e integração do HCA G2 com sistemas de gestão 
  • FIAP_EV Challenge_2026_Mentoria1.pdf p.5
  • FIAP_EV Challenge_2026_Mentoria1.pdf p.5


###**P3**:me fale sobre o protocolo OCPP 1.6

In [ ]:
# Mude a pergunta abaixo para testar diferentes assuntos sobre a GoodWe
pergunta_teste = 'me fale sobre o protocolo OCPP 1.6 '

# Executa o pipeline completo: busca no banco vetorial → geração da resposta
resultado = graph.invoke({'question': pergunta_teste})

# Exibe a pergunta e a resposta gerada
print(f'Pergunta: {pergunta_teste}')
print(f'\nResposta: {resultado["answer"]}')

# Exibe de quais documentos vieram os trechos usados para gerar a resposta
print('\nFontes usadas para responder:')
for doc in resultado['context']:
    fonte  = doc.metadata.get('source', 'desconhecida')  # URL ou caminho do arquivo de origem
    pagina = doc.metadata.get('page', '')                # número da página (só para PDFs)
    print(f'  • {os.path.basename(str(fonte))} {"p." + str(pagina) if pagina != "" else ""}')

Pergunta: me fale sobre o protocolo OCPP 1.6 

Resposta: O OCPP 1.6, lançado em 2015, trouxe diversas melhorias e é amplamente implementado por fabricantes de estações de recarga (CS) e sistemas de gerenciamento de estações de recarga (CSMS) ao redor do mundo. As principais funções do OCPP 1.6 incluem:

- Suporte para versões SOAP e JSON
- Suporte para carregamento inteligente, incluindo balanceamento de carga e uso de perfis de carga
- Suporte para gerenciamento de listas (local)
- Status adicionais
- Solicitações de envio de mensagens, como tempo ou status do ponto de carregamento (CP)
- Melhorias menores nas especificações em comparação com o OCPP 1.5

Essas funcionalidades têm contribuído para a crescente adoção do protocolo entre os fabricantes.

Fontes usadas para responder:
  •  
  •  
  • ?p=16 
  • ?p=16 


###**P3**:Qual é a potência máxima de carregamento do HCA G2 e quais modos de carga ele suporta?

In [ ]:
# Mude a pergunta abaixo para testar diferentes assuntos sobre a GoodWe
pergunta_teste = 'Qual é a potência máxima de carregamento do HCA G2 e quais modos de carga ele suporta?'

# Executa o pipeline completo: busca no banco vetorial → geração da resposta
resultado = graph.invoke({'question': pergunta_teste})

# Exibe a pergunta e a resposta gerada
print(f'Pergunta: {pergunta_teste}')
print(f'\nResposta: {resultado["answer"]}')

# Exibe de quais documentos vieram os trechos usados para gerar a resposta
print('\nFontes usadas para responder:')
for doc in resultado['context']:
    fonte  = doc.metadata.get('source', 'desconhecida')  # URL ou caminho do arquivo de origem
    pagina = doc.metadata.get('page', '')                # número da página (só para PDFs)
    print(f'  • {os.path.basename(str(fonte))} {"p." + str(pagina) if pagina != "" else ""}')

Pergunta: Qual é a potência máxima de carregamento do HCA G2 e quais modos de carga ele suporta?

Resposta: A potência máxima de carregamento do Carregador Veicular GoodWe HCA G2 é de 7 kW para a versão monofásica e de 11/22 kW para a versão trifásica. Ele suporta múltiplos modos de carga, que são configuráveis pelo usuário e gerenciáveis através do aplicativo SEMS+ da GoodWe.

Fontes usadas para responder:
  • GW_HCA G2_Datasheet-PT.pdf p.0
  • GW_HCA G2_Datasheet-PT.pdf p.0
  • GW_HCA G2_Datasheet-PT.pdf p.0
  • GW_HCA G2_User Manual-PT.pdf p.0


###**P4**:É possível limitar a potência de carregamento por usuário no HCA G2 para evitar sobrecarga na rede do condomínio?

In [ ]:
# Mude a pergunta abaixo para testar diferentes assuntos sobre a GoodWe
pergunta_teste = 'É possível limitar a potência de carregamento por usuário no HCA G2 para evitar sobrecarga na rede do condomínio? '

# Executa o pipeline completo: busca no banco vetorial → geração da resposta
resultado = graph.invoke({'question': pergunta_teste})

# Exibe a pergunta e a resposta gerada
print(f'Pergunta: {pergunta_teste}')
print(f'\nResposta: {resultado["answer"]}')

# Exibe de quais documentos vieram os trechos usados para gerar a resposta
print('\nFontes usadas para responder:')
for doc in resultado['context']:
    fonte  = doc.metadata.get('source', 'desconhecida')  # URL ou caminho do arquivo de origem
    pagina = doc.metadata.get('page', '')                # número da página (só para PDFs)
    print(f'  • {os.path.basename(str(fonte))} {"p." + str(pagina) if pagina != "" else ""}')

Pergunta: É possível limitar a potência de carregamento por usuário no HCA G2 para evitar sobrecarga na rede do condomínio? 

Resposta: Sim, é possível limitar a potência de carregamento no HCA G2 para evitar sobrecarga na rede do condomínio. Ao ativar o controle dinâmico de carga, o carregador ajusta a velocidade de carregamento ou até pausa o carregamento com base nos dados do medidor e na corrente de conexão à rede definida, evitando o disparo do fusível principal. Essa configuração pode ser feita via SolarGo ou SEMS+.

Fontes usadas para responder:
  • FIAP_EV Challenge_2026_Mentoria1.pdf p.6
  • FIAP_EV Challenge_2026_Mentoria1.pdf p.6
  • GW_HCA G2_User Manual-PT.pdf p.48
  • GW_HCA G2_User Manual-PT.pdf p.48


###**P5**:Qual é o grau de proteção IP do carregador HCA G2 e onde ele pode ser instalado?

In [ ]:
# Mude a pergunta abaixo para testar diferentes assuntos sobre a GoodWe
pergunta_teste = 'Qual é o grau de proteção IP do carregador HCA G2 e onde ele pode ser instalado?'

# Executa o pipeline completo: busca no banco vetorial → geração da resposta
resultado = graph.invoke({'question': pergunta_teste})

# Exibe a pergunta e a resposta gerada
print(f'Pergunta: {pergunta_teste}')
print(f'\nResposta: {resultado["answer"]}')

# Exibe de quais documentos vieram os trechos usados para gerar a resposta
print('\nFontes usadas para responder:')
for doc in resultado['context']:
    fonte  = doc.metadata.get('source', 'desconhecida')  # URL ou caminho do arquivo de origem
    pagina = doc.metadata.get('page', '')                # número da página (só para PDFs)
    print(f'  • {os.path.basename(str(fonte))} {"p." + str(pagina) if pagina != "" else ""}')

Pergunta: Qual é o grau de proteção IP do carregador HCA G2 e onde ele pode ser instalado?

Resposta: O grau de proteção IP do carregador HCA G2 é IP55. Ele pode ser instalado tanto na parede quanto no piso, utilizando um suporte opcional. Lembre-se também de que é necessária a proteção RCD tipo A externo.

Fontes usadas para responder:
  • GW_HCA G2_Datasheet-PT.pdf p.1
  • GW_HCA G2_Datasheet-PT.pdf p.1
  • GW_HCA G2_Datasheet-PT.pdf p.1
  • GW_HCA G2_Datasheet-AU.pdf p.1


###**P6**:Quais certificações internacionais o HCA G2 possui?

In [ ]:
# Mude a pergunta abaixo para testar diferentes assuntos sobre a GoodWe
pergunta_teste = 'Quais certificações internacionais o HCA G2 possui?'

# Executa o pipeline completo: busca no banco vetorial → geração da resposta
resultado = graph.invoke({'question': pergunta_teste})

# Exibe a pergunta e a resposta gerada
print(f'Pergunta: {pergunta_teste}')
print(f'\nResposta: {resultado["answer"]}')

# Exibe de quais documentos vieram os trechos usados para gerar a resposta
print('\nFontes usadas para responder:')
for doc in resultado['context']:
    fonte  = doc.metadata.get('source', 'desconhecida')  # URL ou caminho do arquivo de origem
    pagina = doc.metadata.get('page', '')                # número da página (só para PDFs)
    print(f'  • {os.path.basename(str(fonte))} {"p." + str(pagina) if pagina != "" else ""}')

Pergunta: Quais certificações internacionais o HCA G2 possui?

Resposta: O GoodWe HCA G2 possui as seguintes certificações internacionais:

- **CE (Conformité Européenne)**: certifica a conformidade com as diretivas de segurança elétrica e compatibilidade eletromagnética (EMC) da União Europeia.

- **CB (IEC System for Mutual Recognition of Testing and Certification)**: certificação reconhecida em mais de 50 países, que atesta a conformidade com normas internacionais de segurança elétrica.

- **RCM (Regulatory Compliance Mark)**: certificação para Austrália e Nova Zelândia, que abrange segurança elétrica e compatibilidade eletromagnética.

- **RETIE (Reglamento Técnico de Instalaciones Eléctricas)**: certificação colombiana exigida para instalações elétricas no mercado da Colômbia.

- **EN 303 645**: norma europeia de cibersegurança para dispositivos conectados à internet (IoT), garantindo boas práticas de segurança cibernética.

Essas certificações garantem a conformidade do HCA G2 co

###**P7**:Como início uma sessão de carga com o GoodWe HCA G2?

In [ ]:
# Mude a pergunta abaixo para testar diferentes assuntos sobre a GoodWe
pergunta_teste = 'Como início uma sessão de carga com o GoodWe HCA G2?'

# Executa o pipeline completo: busca no banco vetorial → geração da resposta
resultado = graph.invoke({'question': pergunta_teste})

# Exibe a pergunta e a resposta gerada
print(f'Pergunta: {pergunta_teste}')
print(f'\nResposta: {resultado["answer"]}')

# Exibe de quais documentos vieram os trechos usados para gerar a resposta
print('\nFontes usadas para responder:')
for doc in resultado['context']:
    fonte  = doc.metadata.get('source', 'desconhecida')  # URL ou caminho do arquivo de origem
    pagina = doc.metadata.get('page', '')                # número da página (só para PDFs)
    print(f'  • {os.path.basename(str(fonte))} {"p." + str(pagina) if pagina != "" else ""}')

Pergunta: Como início uma sessão de carga com o GoodWe HCA G2?

Resposta: Para iniciar uma sessão de carga com o GoodWe HCA G2, você pode fazê-lo das seguintes maneiras:

1. **Cartão RFID**: Utilize um cartão RFID para autenticação e iniciar o carregamento.
   
2. **Aplicativo**: Inicie a sessão de carga através do aplicativo compatível.

3. **Conexão do Plugue**: A sessão de carregamento também pode ser iniciada automaticamente ao conectar o plugue de carregamento.

Essas opções garantem a flexibilidade e conveniência durante o processo de carregamento.

Fontes usadas para responder:
  • FAQ — Protocolo OCPP e integração do HCA G2 com sistemas de gestão 
  • FAQ — Protocolo OCPP e integração do HCA G2 com sistemas de gestão 
  • GW_HCA G2_User Manual-PT.pdf p.9
  • GW_HCA G2_User Manual-PT.pdf p.9


###**P8**:E como eu sei quando o carro terminou de carregar?

In [ ]:
# Mude a pergunta abaixo para testar diferentes assuntos sobre a GoodWe
pergunta_teste = 'E como eu sei quando o carro terminou de carregar?'

# Executa o pipeline completo: busca no banco vetorial → geração da resposta
resultado = graph.invoke({'question': pergunta_teste})

# Exibe a pergunta e a resposta gerada
print(f'Pergunta: {pergunta_teste}')
print(f'\nResposta: {resultado["answer"]}')

# Exibe de quais documentos vieram os trechos usados para gerar a resposta
print('\nFontes usadas para responder:')
for doc in resultado['context']:
    fonte  = doc.metadata.get('source', 'desconhecida')  # URL ou caminho do arquivo de origem
    pagina = doc.metadata.get('page', '')                # número da página (só para PDFs)
    print(f'  • {os.path.basename(str(fonte))} {"p." + str(pagina) if pagina != "" else ""}')

Pergunta: E como eu sei quando o carro terminou de carregar?

Resposta: Você pode saber quando o carro terminou de carregar pelo seguinte:

1. **Indicador LED**: Quando a carga é concluída, o LED do HCA G2 muda para **VERDE FIXO**.
2. **Notificação digital via OCPP**: Se o carregador estiver integrado a um aplicativo ou portal de gestão, você receberá uma notificação digital com informações da sessão, como energia total fornecida, duração da sessão e custo (se o sistema de billing estiver configurado).
3. **O próprio veículo**: A maioria dos veículos elétricos para de aceitar carga automaticamente quando a bateria atinge 100%, o que coincide com a indicação de LED verde fixo no carregador. 

Essas são as formas de identificar o término da sessão de carregamento.

Fontes usadas para responder:
  • FAQ — LED e notificação de fim de carregamento — HCA G2 
  • FAQ — LED e notificação de fim de carregamento — HCA G2 
  • FAQ — LED e notificação de fim de carregamento — HCA G2 
  • FAQ — LED

###**P9**:Quais carregadores da frota estão com falha ou offline atualmente?

In [ ]:
# Mude a pergunta abaixo para testar diferentes assuntos sobre a GoodWe
pergunta_teste = 'Quais carregadores da frota estão com falha ou offline atualmente?'

# Executa o pipeline completo: busca no banco vetorial → geração da resposta
resultado = graph.invoke({'question': pergunta_teste})

# Exibe a pergunta e a resposta gerada
print(f'Pergunta: {pergunta_teste}')
print(f'\nResposta: {resultado["answer"]}')

# Exibe de quais documentos vieram os trechos usados para gerar a resposta
print('\nFontes usadas para responder:')
for doc in resultado['context']:
    fonte  = doc.metadata.get('source', 'desconhecida')  # URL ou caminho do arquivo de origem
    pagina = doc.metadata.get('page', '')                # número da página (só para PDFs)
    print(f'  • {os.path.basename(str(fonte))} {"p." + str(pagina) if pagina != "" else ""}')

Pergunta: Quais carregadores da frota estão com falha ou offline atualmente?

Resposta: Não tenho essa informação nos documentos disponíveis.

Fontes usadas para responder:
  • GW_HCA G2_User Manual-PT.pdf p.65
  • GW_HCA G2_User Manual-PT.pdf p.65
  • GW_HCA G2_User Manual-PT.pdf p.65
  • GW_HCA G2_User Manual-PT.pdf p.64


###**P10**:Qual foi o consumo médio mensal dos carregadores instalados nos últimos 3 meses?

In [ ]:
# Mude a pergunta abaixo para testar diferentes assuntos sobre a GoodWe
pergunta_teste = 'Qual foi o consumo médio mensal dos carregadores instalados nos últimos 3 meses?'

# Executa o pipeline completo: busca no banco vetorial → geração da resposta
resultado = graph.invoke({'question': pergunta_teste})

# Exibe a pergunta e a resposta gerada
print(f'Pergunta: {pergunta_teste}')
print(f'\nResposta: {resultado["answer"]}')

# Exibe de quais documentos vieram os trechos usados para gerar a resposta
print('\nFontes usadas para responder:')
for doc in resultado['context']:
    fonte  = doc.metadata.get('source', 'desconhecida')  # URL ou caminho do arquivo de origem
    pagina = doc.metadata.get('page', '')                # número da página (só para PDFs)
    print(f'  • {os.path.basename(str(fonte))} {"p." + str(pagina) if pagina != "" else ""}')

Pergunta: Qual foi o consumo médio mensal dos carregadores instalados nos últimos 3 meses?

Resposta: Não tenho essa informação nos documentos disponíveis.

Fontes usadas para responder:
  • GW_HCA G2_User Manual-PT.pdf p.12
  • GW_HCA G2_User Manual-PT.pdf p.12
  • GW_HCA G2_User Manual-PT.pdf p.12
  • GW_HCA G2_User Manual-PT.pdf p.13


## 10. Chatbot Interativo com Histórico de Conversa

In [ ]:
import re
import json
import uuid
import os
import logging
from datetime import datetime
from google.colab import auth#, drive
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from openai import OpenAI

# Suprime os warnings do httplib2
logging.getLogger('google_auth_httplib2').setLevel(logging.ERROR)
logging.getLogger('googleapiclient.discovery_cache').setLevel(logging.ERROR)

# ── Autenticação e configuração do Drive ──────────────────────────────────────

#drive.mount('/content/drive')
auth.authenticate_user()
drive_service = build('drive', 'v3', cache_discovery=False)

FOLDER_ID  = '1joGNZlsAB59zkfeUVZrOWNybeYPe8rXs'
LOCAL_FILE = '/content/conversas_chatbot.jsonl'
DRIVE_NAME = 'conversas_chatbot.jsonl'

client_openai = OpenAI(api_key=os.environ['OPENAI_API_KEY'])

SYSTEM_PROMPT = (
    'Você é um assistente especializado no carregador veicular GoodWe HCA G2 '
    'e nos produtos GoodWe em geral. '
    'Você atende cinco perfis de usuário:'
    'Operadores comerciais de eletropostos (ChargeGrid Intelligence)'
    'Síndicos e administradores de condomínios (EV ChargeOps)'
    'Moradores e usuários finais do carregador'
    'Técnicos de instalação e manutenção'
    'GoodWe (monitoramento centralizado e gestão da frota de dispositivos alugados)'
    'Responda usando APENAS as informações do contexto fornecido. '
    'Não invente dados técnicos.'
    'Se não souber a resposta com base no contexto, diga '
    '"Não tenho essa informação nos documentos disponíveis". '
    'Responda sempre em português brasileiro de forma clara e objetiva.\n\n'
    'Para perguntas técnicas de instalação, diga '
    '"Consulte um eletricista habilitado e siga as normas locais."'
    'Mantenha o contexto da conversa — '
    'use o histórico de mensagens para responder perguntas de acompanhamento.'
)


# ── Validações ────────────────────────────────────────────────────────────────

def validar_email(email: str) -> bool:
    return bool(re.match(r'^[\w\.-]+@[\w\.-]+\.\w{2,}$', email))


def validar_cpf(cpf: str) -> bool:
    cpf = re.sub(r'\D', '', cpf)
    if len(cpf) != 11 or len(set(cpf)) == 1:
        return False
    def digito(cpf, peso):
        soma = sum(int(d) * p for d, p in zip(cpf, range(peso, 1, -1)))
        resto = soma % 11
        return 0 if resto < 2 else 11 - resto
    return int(cpf[9]) == digito(cpf, 10) and int(cpf[10]) == digito(cpf, 11)


def coletar_email() -> str:
    while True:
        email = input('E-mail: ').strip()
        if validar_email(email):
            return email
        print('  ⚠ E-mail inválido. Exemplo: nome@email.com')


def coletar_cpf() -> str:
    while True:
        cpf = input('CPF (somente números ou 000.000.000-00): ').strip()
        if validar_cpf(cpf):
            return re.sub(r'\D', '', cpf)
        print('  ⚠ CPF inválido. Verifique os dígitos e tente novamente.')


def perguntar_sim_nao(mensagem: str) -> str:
    """Pergunta sim/não e repete até receber uma resposta válida."""
    while True:
        resposta = input(mensagem).strip().lower()
        if resposta in ['sim', 's']:
            return 'sim'
        if resposta in ['não', 'nao', 'n']:
            return 'nao'
        print('  ⚠ Não entendi sua resposta. Por favor, digite "sim" ou "não".')


# ── Salvamento no Drive via API ───────────────────────────────────────────────

def registrar_conversa(sessao: dict):
    """Acrescenta a sessão ao arquivo local e sincroniza com o Drive."""
    with open(LOCAL_FILE, 'a', encoding='utf-8') as f:
        f.write(json.dumps(sessao, ensure_ascii=False) + '\n')

    resultado = drive_service.files().list(
        q=f"name='{DRIVE_NAME}' and '{FOLDER_ID}' in parents and trashed=false",
        fields='files(id)'
    ).execute()
    arquivos = resultado.get('files', [])

    media = MediaFileUpload(LOCAL_FILE, mimetype='text/plain', resumable=False)

    if arquivos:
        drive_service.files().update(
            fileId=arquivos[0]['id'],
            media_body=media
        ).execute()
    else:
        drive_service.files().create(
            body={'name': DRIVE_NAME, 'parents': [FOLDER_ID]},
            media_body=media
        ).execute()

    #print('✅ Conversa salva no Google Drive.')


# ── RAG ───────────────────────────────────────────────────────────────────────

def buscar_contexto(pergunta: str) -> str:
    docs_relevantes = vector_store.similarity_search(pergunta, k=3)
    return '\n\n'.join(doc.page_content for doc in docs_relevantes)


# ── Continuidade ──────────────────────────────────────────────────────────────

def perguntar_continuidade(nome_exibicao: str):
    """
    Retorna ('encerrar', None) ou ('continuar', texto_digitado).
    Qualquer coisa que não seja 'encerrar' vira a próxima pergunta.
    """
    print('\n' + '─' * 60)
    print(f'  {nome_exibicao}, posso te ajudar com mais alguma coisa?')
    print('─' * 60)
    print('  ↩  Pressione ENTER para digitar uma nova pergunta')
    print('  ✖  Digite  "encerrar"  para finalizar o chat')
    print('─' * 60)
    digitado = input('  Sua próxima pergunta (ou "encerrar"): ').strip()

    if digitado.lower() == 'encerrar':
        return 'encerrar', None
    return 'continuar', digitado  # devolve o que foi digitado como próxima pergunta


# ── Coleta de dados ───────────────────────────────────────────────────────────

def coletar_dados_usuario():
    print('=' * 60)
    print('   Chatbot GoodWe — FIAP EV Challenge 2026')
    print('=' * 60)
    print(
        '\nOlá! Bem-vindo ao suporte GoodWe.\n'
        '\nGostaríamos de convidar você a compartilhar alguns dados para '
        'acompanhamento da sua solicitação e para um atendimento mais '
        'personalizado. Caso aceite, pedimos: nome, celular, e-mail e CPF.\n'
        '\n🔒 LGPD: Seus dados são coletados com base na Lei nº 13.709/2018 '
        '(Lei Geral de Proteção de Dados). Serão usados exclusivamente para '
        'este atendimento, armazenados com segurança e não compartilhados '
        'com terceiros sem seu consentimento.\n'
    )

    resposta = perguntar_sim_nao('Deseja compartilhar seus dados? (sim/não): ')

    dados_usuario = {
        'compartilhou_dados': False,
        'nome': None,
        'celular': None,
        'email': None,
        'cpf': None,
    }

    if resposta == 'sim':
        dados_usuario['compartilhou_dados'] = True
        dados_usuario['nome']    = input('\nNome completo: ').strip()
        dados_usuario['celular'] = input('Celular (com DDD): ').strip()
        dados_usuario['email']   = coletar_email()
        dados_usuario['cpf']     = coletar_cpf()
        nome_exibicao = dados_usuario['nome'].split()[0]
    else:
        apelido = input('\nSem problemas! Como posso chamá-lo(a)? ').strip()
        dados_usuario['nome'] = apelido
        nome_exibicao = apelido

    print(f'\nOi, {nome_exibicao}! Em que posso te ajudar?')
    print('-' * 60)
    print('Após cada resposta, você poderá digitar uma nova pergunta ou encerrar.')
    print('-' * 60)

    return dados_usuario, nome_exibicao


# ── Chatbot principal ─────────────────────────────────────────────────────────

def chatbot_rag():
    dados_usuario, nome_exibicao = coletar_dados_usuario()

    historico = [{'role': 'system', 'content': SYSTEM_PROMPT}]

    sessao = {
        'sessao_id': str(uuid.uuid4()),
        'inicio': datetime.now().isoformat(),
        'usuario': dados_usuario,
        'turnos': [],
        'encerrado_por': 'usuario',
    }

    proxima_pergunta = None  # guarda pergunta digitada na tela de continuidade

    while True:
        # Usa a pergunta já digitada na tela de continuidade, se houver
        if proxima_pergunta:
            pergunta = proxima_pergunta
            proxima_pergunta = None
            print(f'\n{nome_exibicao}: {pergunta}')
        else:
            pergunta = input(f'\n{nome_exibicao}: ').strip()

        if not pergunta:
            continue

        contexto = buscar_contexto(pergunta)
        mensagem_usuario = f'Contexto dos documentos:\n{contexto}\n\nPergunta: {pergunta}'
        historico.append({'role': 'user', 'content': mensagem_usuario})

        try:
            resposta = client_openai.chat.completions.create(
                model='gpt-4o-mini',
                messages=historico,
                max_tokens=600,
                temperature=0.3,
            )
            texto_resposta = resposta.choices[0].message.content

        except Exception as e:
            texto_resposta = f'Erro na API: {e}'

        print(f'\nAssistente: {texto_resposta}')

        sessao['turnos'].append({
            'timestamp': datetime.now().isoformat(),
            'pergunta': pergunta,
            'resposta': texto_resposta,
        })

        historico[-1] = {'role': 'user', 'content': pergunta}
        historico.append({'role': 'assistant', 'content': texto_resposta})

        if len(historico) > 21:
            historico = [historico[0]] + historico[-20:]

        escolha, proxima_pergunta = perguntar_continuidade(nome_exibicao)
        if escolha == 'encerrar':
            print(f'\nAté logo, {nome_exibicao}! Obrigado por entrar em contato com o suporte GoodWe.')
            sessao['encerrado_por'] = 'fluxo_encerramento'
            sessao['fim'] = datetime.now().isoformat()
            registrar_conversa(sessao)
            break

    if 'fim' not in sessao:
        sessao['fim'] = datetime.now().isoformat()
        registrar_conversa(sessao)


chatbot_rag()

   Chatbot GoodWe — FIAP EV Challenge 2026

Olá! Bem-vindo ao suporte GoodWe.

Gostaríamos de convidar você a compartilhar alguns dados para acompanhamento da sua solicitação e para um atendimento mais personalizado. Caso aceite, pedimos: nome, celular, e-mail e CPF.

🔒 LGPD: Seus dados são coletados com base na Lei nº 13.709/2018 (Lei Geral de Proteção de Dados). Serão usados exclusivamente para este atendimento, armazenados com segurança e não compartilhados com terceiros sem seu consentimento.

Deseja compartilhar seus dados? (sim/não): ssim
  ⚠ Não entendi sua resposta. Por favor, digite "sim" ou "não".
Deseja compartilhar seus dados? (sim/não): sim

Nome completo: bru
Celular (com DDD): 1651
E-mail: Teste@gmail.com
CPF (somente números ou 000.000.000-00): 550.111.555.-15
  ⚠ CPF inválido. Verifique os dígitos e tente novamente.
CPF (somente números ou 000.000.000-00): 228.509.588-08

Oi, bru! Em que posso te ajudar?
------------------------------------------------------------
Após